# Explorador de pistas

Compará dos configuraciones del generador de pistas de CarRacing en las mismas 3 seeds. Los sliders arrancan con las dos distribuciones de `tp_rl.ipynb` (caso 1 = original, caso 2 = nueva).

| Slider | Variable | Qué cambia |
|-|-|-|
| paso | `TRACK_DETAIL_STEP` (factor sobre CarRacing-v3) | Más chico = curvas más cerradas. Debajo de ~0.25 la mayoría de las pistas no cierran |
| giro máx. | `TRACK_TURN_RATE` (valor absoluto, CarRacing-v3: 0.31) | Tope del giro por paso. Solo limita si el paso es chico |
| ancho | `TRACK_WIDTH` (factor) | Ancho de la pista |
| radio | `TRACK_RAD` (factor) | Tamaño del circuito. Más chico = vuelta más corta y menos tiles (render más rápido) |

Debajo de cada fila: radio de giro mínimo, ancho, largo, tiles e intentos del generador por pista (promedio de las 3 seeds). Con muchos intentos, el reset de cada episodio se vuelve lento. El botón imprime los parámetros para copiar en `carracing_env.py`.

Ejecutá todas las celdas (en Jupyter Notebook o JupyterLab) para ver los sliders.

In [2]:
import sys

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from gymnasium.envs.box2d import car_racing
from IPython.display import display
from matplotlib.collections import PolyCollection

sys.path.insert(0, ".")
from carracing_env import NEW, ORIGINAL, CustomTrackCarRacing

VANILLA = {name: getattr(car_racing, name) for name in ["TRACK_DETAIL_STEP", "TRACK_TURN_RATE", "TRACK_WIDTH", "TRACK_RAD"]}
FACTOR_VARIABLES = ["TRACK_DETAIL_STEP", "TRACK_WIDTH", "TRACK_RAD"]


class CountingCarRacing(CustomTrackCarRacing):
    """Counts how many times the generator runs, failed tries included."""

    def _create_track(self):
        self.tries += 1
        return super()._create_track()


def generate(params, seed):
    env = CountingCarRacing([params], continuous=False)
    env.tries = 0
    env.reset(seed=seed)
    return env


def describe(params, envs):
    step = params["TRACK_DETAIL_STEP"]
    curvatures, lengths = [], []
    for env in envs:
        track = np.array(env.track)
        curvatures.append(np.percentile(np.abs(np.diff(np.unwrap(track[:, 1]))) / step, 99))
        lengths.append(np.linalg.norm(np.diff(track[:, 2:4], axis=0), axis=1).sum())
    tiles = np.mean([len(env.track) for env in envs])
    tries = np.mean([env.tries for env in envs])
    return (f"radio mín. {1 / np.mean(curvatures):.1f} · ancho {2 * params['TRACK_WIDTH']:.1f} · "
            f"largo {np.mean(lengths):.0f} · {tiles:.0f} tiles · {tries:.1f} intentos por pista")

In [3]:
def case_sliders(params):
    """Sliders that start from a parameter set of carracing_env."""
    full = {**VANILLA, **params}
    factor = lambda name: full[name] / VANILLA[name]
    style = dict(continuous_update=False, style={"description_width": "70px"}, layout=widgets.Layout(width="340px"))
    return dict(
        TRACK_DETAIL_STEP=widgets.FloatSlider(value=factor("TRACK_DETAIL_STEP"), min=0.2, max=8, step=0.05, description="paso", **style),
        TRACK_TURN_RATE=widgets.FloatSlider(value=full["TRACK_TURN_RATE"], min=0.05, max=3, step=0.05, description="giro máx.", **style),
        TRACK_WIDTH=widgets.FloatSlider(value=factor("TRACK_WIDTH"), min=0.3, max=2.5, step=0.05, description="ancho", **style),
        TRACK_RAD=widgets.FloatSlider(value=factor("TRACK_RAD"), min=0.3, max=1.5, step=0.05, description="radio", **style),
    )


def to_params(sliders):
    return {name: s.value * VANILLA[name] if name in FACTOR_VARIABLES else s.value for name, s in sliders.items()}


cases = {"caso 1": case_sliders(ORIGINAL), "caso 2": case_sliders(NEW)}
seeds = [widgets.IntText(value=s, description=f"seed {i + 1}", layout=widgets.Layout(width="160px")) for i, s in enumerate([0, 1, 2])]
plot = widgets.Output()
printed = widgets.Output()


def redraw(_=None):
    with plot:
        plot.clear_output(wait=True)
        fig, axs = plt.subplots(len(cases), len(seeds), figsize=(13, 3.6 * len(cases)), sharex=True, sharey=True)
        for row, (label, sliders) in enumerate(cases.items()):
            params = to_params(sliders)
            envs = [generate(params, s.value) for s in seeds]
            for col, env in enumerate(envs):
                polygons = [p for p, _ in env.road_poly]
                colors = [np.array(c) / 255 for _, c in env.road_poly]
                axs[row, col].add_collection(PolyCollection(polygons, facecolors=colors))
                axs[row, col].set_aspect("equal")
                axs[row, col].set_title(f"{label}, seed {seeds[col].value}", fontsize=9)
                env.close()
            axs[row, 1].text(0.5, -0.1, describe(params, envs), transform=axs[row, 1].transAxes, ha="center", va="top", fontsize=9)
        axs[0, 0].autoscale()
        fig.tight_layout()
        plt.show()


def print_params(_):
    with printed:
        printed.clear_output()
        for label, sliders in cases.items():
            factors = ", ".join(f"{name}=car_racing.{name} * {s.value:g}" if name in FACTOR_VARIABLES else f"{name}={s.value:g}"
                                for name, s in sliders.items())
            print(f"{label}: dict({factors})")


for widget in [*seeds, *(s for sliders in cases.values() for s in sliders.values())]:
    widget.observe(redraw, names="value")
button = widgets.Button(description="Imprimir parámetros")
button.on_click(print_params)

columns = [widgets.VBox([widgets.HTML(f"<b>{label}</b>"), *sliders.values()]) for label, sliders in cases.items()]
display(widgets.VBox([widgets.HBox(seeds), widgets.HBox(columns), button, printed, plot]))
redraw()